# 5.11 · CatBoost

> **课程定位 / Where this fits**
> boosting 三巨头收官。CatBoost(Yandex)的杀手锏是**类别特征**: 用**有序目标编码(ordered target statistics)**避免目标泄漏(3.9), 配合**对称(oblivious)树**和**有序提升(ordered boosting)**减少预测偏移。类别特征多、又不想手工编码时, 它默认参数就很能打。
> CatBoost's edge is principled categorical handling via ordered target statistics (no leakage), plus symmetric trees and ordered boosting. Strong out-of-the-box on category-heavy data.

> 💡 **面试相关 / Interview-relevant**
> - "目标编码为什么会泄漏 / CatBoost 怎么解决" ★★★★★（有序TS）
> - "ordered boosting 解决什么(prediction shift)" ★★★★
> - "对称树是什么 / 有什么好处" ★★★★
> - "CatBoost vs LightGBM 类别处理区别" ★★★★

---

## 学习目标 / Learning Objectives
1. 普通目标编码的**泄漏**问题(回顾 3.9/3.5)。
2. **有序目标统计**如何防泄漏。
3. **有序提升**消除预测偏移。
4. **对称树**结构与好处。
5. 三巨头横向对比收尾。

## 目录 / TOC
1. [目标编码泄漏 → 有序TS ⭐](#1)
2. [有序提升 + 对称树 ⭐](#2)
3. [💰 数据(含类别) + 训练](#3)
4. [类别处理: CatBoost vs one-hot ⭐](#4)
5. [三巨头横向对比](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 目标编码泄漏 → 有序TS ⭐ / Target Leakage → Ordered TS

**目标编码(target encoding, 3.5)**: 把类别值替换成"该类别下目标的平均值"(如 city→该 city 的平均收入)。问题: 算某行的编码时**用到了该行自己的标签** → **目标泄漏(3.9)**, 训练集虚高、测试崩。

**CatBoost 的有序目标统计(Ordered Target Statistics)**: 给样本一个随机"时间顺序", 算第 $i$ 行的类别编码时**只用排在它前面的样本**的标签(像在线学习的因果约束)。这样**绝不看自己的标签**, 从根上防泄漏。公式(带先验平滑):
$$\text{enc}(x_i) = \frac{\sum_{j<i}[x_j=x_i]\,y_j + a\,p}{\sum_{j<i}[x_j=x_i] + a}$$
$a$ 平滑强度, $p$ 全局先验。多组随机排列取平均, 进一步稳。


<a id="2"></a>
## 2. 有序提升 + 对称树 ⭐ / Ordered Boosting & Symmetric Trees

**预测偏移(prediction shift)**: 标准 GBDT 用同一批数据既算残差又训树, 残差和模型有微妙相关 → 训练分布偏离测试分布(又一种泄漏)。**有序提升**: 算样本 $i$ 的梯度时, 用"只在它之前样本上训练"的模型, 同样的因果约束消除偏移。

**对称(oblivious)树**: CatBoost 的基学习器——**整层用同一个(特征, 阈值)分裂**, 长成完全平衡的树。好处: (1) 强正则, 抗过拟合; (2) 预测极快(可向量化成位运算); (3) 默认参数就稳。代价是单棵树表达力弱, 但 boosting 叠加足够。


<a id="3"></a>
## 3. 数据(含类别) + 训练 / Data with Categoricals

复用 5.8 的**合成 Adult Income**, 但这次**加入真正有信号的类别特征** `occupation`(不同职业有不同高收入倾向), 好展示 CatBoost 的类别处理威力。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid")

def make_income_cat(n=12000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n); edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    occ = rng.choice(["tech","mgmt","sales","admin","service"], n, p=[.2,.15,.25,.2,.2])
    occ_effect = {"tech":1.2, "mgmt":1.5, "sales":0.2, "admin":-0.3, "service":-0.8}
    bonus = np.array([occ_effect[o] for o in occ])    # 职业带真实信号
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years + 0.02*hours
             + 0.0002*np.sqrt(capital_gain)*edu_years*0.3 + bonus + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours,
                      "capital_gain": capital_gain.round(0), "occupation": occ})
    return X, y

X, y = make_income_cat()
print(f"合成 Adult Income(含类别): {X.shape}, 高收入率 {y.mean():.0%}")
print("各职业高收入率:"); print(pd.crosstab(X.occupation, y, normalize="index").round(2).to_string())
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


In [ ]:
cat_features = ["occupation"]
clf = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6,
                         cat_features=cat_features, verbose=0, random_seed=0)
clf.fit(X_tr, y_tr)
print(f"CatBoost(原生类别) test AUC: {roc_auc_score(y_te, clf.predict_proba(X_te)[:,1]):.4f}")
print("直接传 cat_features, CatBoost 内部用有序目标统计编码, 无需手工处理")


<a id="4"></a>
## 4. 类别处理: CatBoost vs one-hot ⭐ / vs Manual Encoding

对比三种处理 `occupation` 的方式, 看 CatBoost 原生处理 vs 朴素做法。


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import xgboost as xgb

num = ["age","edu_years","hours","capital_gain"]
# (a) 丢掉类别特征
xgb_drop = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                             eval_metric="auc", random_state=0).fit(X_tr[num], y_tr)
auc_drop = roc_auc_score(y_te, xgb_drop.predict_proba(X_te[num])[:,1])

# (b) one-hot + XGBoost
ct = ColumnTransformer([("oh", OneHotEncoder(handle_unknown="ignore"), ["occupation"])], remainder="passthrough")
Xtr_oh = ct.fit_transform(X_tr); Xte_oh = ct.transform(X_te)
xgb_oh = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                           eval_metric="auc", random_state=0).fit(Xtr_oh, y_tr)
auc_oh = roc_auc_score(y_te, xgb_oh.predict_proba(Xte_oh)[:,1])

# (c) CatBoost 原生
auc_cat = roc_auc_score(y_te, clf.predict_proba(X_te)[:,1])

print(f"(a) 丢掉 occupation      AUC: {auc_drop:.4f}")
print(f"(b) one-hot + XGBoost    AUC: {auc_oh:.4f}")
print(f"(c) CatBoost 原生类别     AUC: {auc_cat:.4f}")
print("\n丢掉类别明显更差(信号丢了); one-hot 和 CatBoost 都能用上, CatBoost 免手工编码")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(["丢弃","one-hot\n+XGB","CatBoost\n原生"], [auc_drop, auc_oh, auc_cat],
       color=["gray","tab:blue","tab:green"])
ax.set_ylim(min(auc_drop,auc_oh,auc_cat)-0.01, max(auc_oh,auc_cat)+0.005)
ax.set_ylabel("test AUC"); ax.set_title("类别特征处理方式对比")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 三巨头横向对比 / The Big-Three Compared

| | XGBoost (5.9) | LightGBM (5.10) | CatBoost (5.11) |
|---|---|---|---|
| 树生长 | level-wise(可选 leaf) | **leaf-wise** | **对称(oblivious)** |
| 加速核心 | 二阶泰勒 + hist | 直方图 + GOSS + EFB | 对称树 + GPU |
| 类别特征 | 需编码(新版有原生) | 原生(直方图分割) | **原生(有序TS, 防泄漏)** |
| 防过拟合特色 | λ,γ 正则 | num_leaves 控制 | **有序提升 + 对称树** |
| 默认好用度 | 需调参 | 需调 num_leaves | **默认即强** |
| 强项场景 | 通用霸主 | 大数据/高维 | 类别特征多 |

**实务选择**: 先 LightGBM(快)做基线; 类别多用 CatBoost; 追极致精度三个都调一遍集成。


In [ ]:
import lightgbm as lgb, time
# 同一含类别数据上三巨头快速对比(XGB/LGBM 用 one-hot, CatBoost 原生)
results = {}
t=time.perf_counter(); results["XGBoost(OH)"]=(roc_auc_score(y_te, xgb_oh.predict_proba(Xte_oh)[:,1]), time.perf_counter()-t)

t=time.perf_counter()
lgbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=0, verbose=-1)
Xtr_c = X_tr.assign(occupation=X_tr.occupation.astype("category"))
Xte_c = X_te.assign(occupation=X_te.occupation.astype("category"))
lgbm.fit(Xtr_c, y_tr, categorical_feature=["occupation"])
results["LightGBM(原生)"]=(roc_auc_score(y_te, lgbm.predict_proba(Xte_c)[:,1]), time.perf_counter()-t)
results["CatBoost(原生)"]=(auc_cat, None)

print("三巨头 test AUC(同一含类别数据集):")
for k,(a,_) in results.items(): print(f"  {k:<16} AUC {a:.4f}")
print("本例 CatBoost 略胜——正是其有序目标统计对(单个高信号)类别特征处理更优;")
print("真实数据上三者常在伯仲之间, 选型更多看易用性/类别处理/速度。")


<a id="6"></a>
## 6. 小结 / Summary

```
CatBoost 三大特色:
  有序目标统计(Ordered TS): 编码类别时只用"之前"样本标签 → 防目标泄漏(3.9)
  有序提升(Ordered Boosting): 算梯度用"之前"样本训的模型 → 消预测偏移
  对称(oblivious)树: 整层同一分裂 → 强正则 + 预测极快 + 默认即强
原生类别特征(传 cat_features), 免 one-hot; 类别多时首选
三巨头: XGB(通用) / LGBM(快, 大数据) / CatBoost(类别多, 默认强)
```

### 💡 面试速查
1. **有序目标统计**防目标编码泄漏: 只用排在前面的样本算编码
2. **有序提升**消除 prediction shift(同一数据既算残差又训树的偏差)
3. **对称树**: 整层同一(特征,阈值), 强正则 + 推理快
4. **三巨头对比**: 生长策略(level/leaf/对称)、类别处理、默认易用度
5. 类别特征多→CatBoost; 大数据→LightGBM; 通用→XGBoost

### 下一节
**5.12 LDA & QDA**——回到经典统计判别。用高斯假设直接建模每类分布, LDA 线性边界、QDA 二次边界, 还能降维, 是生成式分类的另一脉。
